# Portfolio Optimization – Efficient Frontier, Sharpe Maximization & Risk Parity

**Author:** Florian Ebner  
**Context:** Quantitative Asset Management – Portfolio Construction & Risk Budgeting

---

## Overview

This notebook implements a complete **Modern Portfolio Theory (MPT)** framework, covering three fundamental portfolio construction approaches used in quantitative asset management:

| Strategy | Goal | Used by |
|----------|------|----------|
| **Mean-Variance Optimization** | Maximize return per unit of risk | Hedge funds, asset managers |
| **Minimum Variance Portfolio** | Minimize total portfolio volatility | Risk-averse mandates, insurers |
| **Maximum Sharpe Ratio** | Best risk-adjusted return | Most institutional investors |
| **Risk Parity** | Equal risk contribution per asset | Bridgewater, pension funds |

### Theoretical Foundation – Markowitz (1952)

Harry Markowitz showed that for a portfolio of N assets, the **expected return** and **variance** are:

$$\mu_p = w^T \mu$$
$$\sigma_p^2 = w^T \Sigma w$$

where $w$ is the weight vector, $\mu$ the expected return vector, and $\Sigma$ the covariance matrix.  
The **Efficient Frontier** is the set of portfolios that maximise return for a given level of risk – no rational investor should hold a portfolio *below* the frontier.

---

## Step 1: Imports and Configuration

We use:
- `numpy` / `pandas` for matrix operations and data handling
- `scipy.optimize` for constrained optimization (this is the mathematical engine)
- `matplotlib` for professional-grade visualizations

In a production setting, returns would be sourced from `yfinance`, Bloomberg, or an internal data warehouse. Here we use **synthetic correlated returns** calibrated to realistic equity market parameters.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.cm as cm
from scipy.optimize import minimize, differential_evolution
from scipy.stats import norm

# ── Reproducibility ───────────────────────────────────────────────────────────
np.random.seed(42)

# ── Simulation Parameters ─────────────────────────────────────────────────────
N_DAYS_HISTORY  = 1260   # ~5 years of trading data
TRADING_DAYS    = 252
RISK_FREE_RATE  = 0.035  # 3.5% annual (approx. current ECB rate)
PORTFOLIO_VALUE = 1_000_000  # €1,000,000

# ── Plot Style ────────────────────────────────────────────────────────────────
plt.rcParams.update({
    'figure.facecolor': 'white',
    'axes.facecolor':   '#f8f9fa',
    'axes.grid':        True,
    'grid.color':       '#e0e0e0',
    'font.family':      'sans-serif',
    'axes.spines.top':  False,
    'axes.spines.right':False,
})

print('Configuration loaded.')
print(f'  Risk-free rate  : {RISK_FREE_RATE*100:.1f}% p.a. (ECB approximation)')
print(f'  History         : {N_DAYS_HISTORY} trading days (~5 years)')
print(f'  Portfolio value : €{PORTFOLIO_VALUE:,.0f}')

## Step 2: Define the Asset Universe and Generate Market Data

We construct a **diversified 8-asset portfolio** spanning multiple sectors and risk profiles – representative of a European institutional equity mandate:

| Asset | Sector | Ann. Return | Ann. Vol |
|-------|--------|-------------|----------|
| SAP   | Technology | 10% | 22% |
| Allianz | Insurance | 8% | 18% |
| Siemens | Industrial | 9% | 20% |
| Deutsche Bank | Financial | 7% | 30% |
| BASF | Chemical | 6% | 24% |
| Vonovia | Real Estate | 5% | 28% |
| RWE | Utilities | 6% | 16% |
| BMW | Consumer | 9% | 26% |

Returns are generated as **correlated multivariate normal** using a realistic correlation structure.

In [ ]:
# ── Asset Universe ────────────────────────────────────────────────────────────
assets = ['SAP', 'Allianz', 'Siemens', 'Deutsche Bank', 'BASF', 'Vonovia', 'RWE', 'BMW']
n      = len(assets)

# Annual parameters
annual_mu  = np.array([0.10, 0.08, 0.09, 0.07, 0.06, 0.05, 0.06, 0.09])
annual_vol = np.array([0.22, 0.18, 0.20, 0.30, 0.24, 0.28, 0.16, 0.26])

# Scale to daily
daily_mu  = annual_mu  / TRADING_DAYS
daily_vol = annual_vol / np.sqrt(TRADING_DAYS)

# ── Correlation Matrix ────────────────────────────────────────────────────────
# Sectors that move together (e.g. Industrials & Consumer, Financials & Insurance)
corr = np.array([
    #SAP   ALV   SIE   DBK   BAS   VNA   RWE   BMW
    [1.00, 0.40, 0.55, 0.35, 0.45, 0.25, 0.20, 0.50],  # SAP
    [0.40, 1.00, 0.50, 0.65, 0.40, 0.45, 0.30, 0.45],  # Allianz
    [0.55, 0.50, 1.00, 0.50, 0.60, 0.35, 0.30, 0.70],  # Siemens
    [0.35, 0.65, 0.50, 1.00, 0.45, 0.50, 0.25, 0.45],  # Deutsche Bank
    [0.45, 0.40, 0.60, 0.45, 1.00, 0.35, 0.40, 0.55],  # BASF
    [0.25, 0.45, 0.35, 0.50, 0.35, 1.00, 0.40, 0.30],  # Vonovia
    [0.20, 0.30, 0.30, 0.25, 0.40, 0.40, 1.00, 0.25],  # RWE
    [0.50, 0.45, 0.70, 0.45, 0.55, 0.30, 0.25, 1.00],  # BMW
])

# Build covariance matrix: Σ_ij = ρ_ij * σ_i * σ_j
cov_daily  = np.outer(daily_vol, daily_vol) * corr
cov_annual = cov_daily * TRADING_DAYS   # annualised covariance

# ── Simulate 5 Years of Daily Returns ────────────────────────────────────────
L = np.linalg.cholesky(cov_daily)
Z = np.random.standard_normal((N_DAYS_HISTORY, n))
daily_returns = daily_mu + Z @ L.T
df_returns    = pd.DataFrame(daily_returns, columns=assets)

# Estimate annualised parameters from simulated history
mu_hat  = df_returns.mean().values * TRADING_DAYS
cov_hat = df_returns.cov().values  * TRADING_DAYS

print('Asset Universe – Estimated Annualised Parameters:')
df_params = pd.DataFrame({
    'Asset':      assets,
    'Ann. Return': mu_hat,
    'Ann. Vol':    np.sqrt(np.diag(cov_hat)),
    'Sharpe':     (mu_hat - RISK_FREE_RATE) / np.sqrt(np.diag(cov_hat))
})
print(df_params.to_string(index=False, float_format=lambda x: f'{x:.4f}'))

## Step 3: Core Portfolio Metrics Functions

Before optimising, we define the fundamental functions that compute portfolio metrics from a weight vector. These are the building blocks used throughout the notebook.

**Portfolio Return:** $\mu_p = w^T \mu$  
**Portfolio Volatility:** $\sigma_p = \sqrt{w^T \Sigma w}$  
**Sharpe Ratio:** $S = \frac{\mu_p - r_f}{\sigma_p}$  
**Maximum Drawdown:** Largest peak-to-trough decline in cumulative returns – a key risk metric for institutional mandates.

In [ ]:
def portfolio_return(w, mu):
    """Annualised expected return of a portfolio."""
    return w @ mu

def portfolio_vol(w, cov):
    """Annualised portfolio volatility (standard deviation)."""
    return np.sqrt(w @ cov @ w)

def sharpe_ratio(w, mu, cov, rf=RISK_FREE_RATE):
    """Annualised Sharpe Ratio."""
    excess = portfolio_return(w, mu) - rf
    vol    = portfolio_vol(w, cov)
    return excess / vol if vol > 1e-10 else 0.0

def max_drawdown(cumulative_returns):
    """
    Maximum Drawdown: largest percentage drop from a peak.
    MDD = max over all t of (peak_before_t - trough_after_peak) / peak_before_t
    """
    rolling_max = np.maximum.accumulate(cumulative_returns)
    drawdowns   = (cumulative_returns - rolling_max) / rolling_max
    return drawdowns.min()   # negative value (loss)

def portfolio_summary(w, mu, cov, label='Portfolio', rf=RISK_FREE_RATE):
    """Print a formatted summary of key portfolio metrics."""
    ret = portfolio_return(w, mu)
    vol = portfolio_vol(w, cov)
    sr  = sharpe_ratio(w, mu, cov, rf)
    print(f'  {label}:')
    print(f'    Return (ann.)  : {ret*100:>7.2f}%')
    print(f'    Volatility     : {vol*100:>7.2f}%')
    print(f'    Sharpe Ratio   : {sr:>7.4f}')
    print(f'    Weights        : {np.round(w, 3)}')

print('Portfolio metric functions defined.')

## Step 4: The Efficient Frontier via Constrained Optimization

The **Efficient Frontier** is constructed by solving a constrained optimization problem for many target return levels:

$$\min_w \; w^T \Sigma w \quad \text{subject to:}$$
$$w^T \mu = \mu_{\text{target}}, \quad \sum_i w_i = 1, \quad w_i \geq 0 \; \forall i$$

The constraint $w_i \geq 0$ enforces **long-only** positions (no short selling) – standard for most institutional mandates.

`scipy.optimize.minimize` with the `SLSQP` (Sequential Least Squares Programming) solver handles this efficiently. We sweep across 200 target return levels to trace the entire frontier.

In [ ]:
def min_variance_for_target(target_return, mu, cov):
    """
    Find the minimum-variance portfolio achieving a given target return.
    Returns the optimized weight vector, or None if infeasible.
    """
    n    = len(mu)
    w0   = np.ones(n) / n   # equal-weight starting point

    constraints = [
        {'type': 'eq', 'fun': lambda w: w.sum() - 1},               # weights sum to 1
        {'type': 'eq', 'fun': lambda w: portfolio_return(w, mu) - target_return},  # target return
    ]
    bounds = [(0.0, 1.0)] * n   # long-only

    result = minimize(
        fun=lambda w: portfolio_vol(w, cov),
        x0=w0,
        method='SLSQP',
        bounds=bounds,
        constraints=constraints,
        options={'ftol': 1e-12, 'maxiter': 1000}
    )
    return result.x if result.success else None

# ── Sweep target returns to trace the Efficient Frontier ─────────────────────
min_ret    = mu_hat.min()
max_ret    = mu_hat.max()
n_points   = 200
target_returns = np.linspace(min_ret + 0.001, max_ret - 0.001, n_points)

frontier_vols    = []
frontier_returns = []
frontier_weights = []

for target in target_returns:
    w = min_variance_for_target(target, mu_hat, cov_hat)
    if w is not None:
        frontier_vols.append(portfolio_vol(w, cov_hat))
        frontier_returns.append(portfolio_return(w, mu_hat))
        frontier_weights.append(w)

frontier_vols    = np.array(frontier_vols)
frontier_returns = np.array(frontier_returns)

print(f'Efficient Frontier computed: {len(frontier_vols)} feasible points.')
print(f'  Volatility range: {frontier_vols.min()*100:.2f}% – {frontier_vols.max()*100:.2f}%')
print(f'  Return range    : {frontier_returns.min()*100:.2f}% – {frontier_returns.max()*100:.2f}%')

## Step 5: Key Optimal Portfolios

Three specific portfolios sit at notable points on the frontier:

1. **Minimum Variance Portfolio (MVP):** The leftmost point of the frontier – minimises volatility regardless of return. Preferred by risk-averse mandates (pension funds, insurers).

2. **Maximum Sharpe Ratio Portfolio (Tangency Portfolio):** The point on the frontier where a line from the risk-free rate is tangent to the frontier. This is the portfolio with the **best risk-adjusted return** – every rational investor with access to the risk-free asset should hold this.

3. **Equal-Weight Portfolio:** The naive 1/N benchmark. Often surprisingly competitive despite its simplicity.

In [ ]:
# ── 1. Minimum Variance Portfolio ─────────────────────────────────────────────
def find_min_variance(mu, cov):
    n  = len(mu)
    w0 = np.ones(n) / n
    constraints = [{'type': 'eq', 'fun': lambda w: w.sum() - 1}]
    bounds = [(0.0, 1.0)] * n
    result = minimize(
        fun=lambda w: portfolio_vol(w, cov),
        x0=w0, method='SLSQP',
        bounds=bounds, constraints=constraints,
        options={'ftol': 1e-12, 'maxiter': 1000}
    )
    return result.x

# ── 2. Maximum Sharpe Ratio Portfolio (Tangency) ──────────────────────────────
def find_max_sharpe(mu, cov, rf=RISK_FREE_RATE):
    n  = len(mu)
    w0 = np.ones(n) / n
    constraints = [{'type': 'eq', 'fun': lambda w: w.sum() - 1}]
    bounds = [(0.0, 1.0)] * n
    result = minimize(
        fun=lambda w: -sharpe_ratio(w, mu, cov, rf),  # minimise negative Sharpe
        x0=w0, method='SLSQP',
        bounds=bounds, constraints=constraints,
        options={'ftol': 1e-12, 'maxiter': 1000}
    )
    return result.x

# ── Compute ───────────────────────────────────────────────────────────────────
w_mvp    = find_min_variance(mu_hat, cov_hat)
w_msr    = find_max_sharpe(mu_hat, cov_hat)
w_equal  = np.ones(n) / n

print('═' * 52)
portfolio_summary(w_mvp,   mu_hat, cov_hat, 'Minimum Variance Portfolio')
print()
portfolio_summary(w_msr,   mu_hat, cov_hat, 'Max Sharpe (Tangency) Portfolio')
print()
portfolio_summary(w_equal, mu_hat, cov_hat, 'Equal-Weight Portfolio')
print('═' * 52)

## Step 6: Risk Parity Portfolio

**Risk Parity** is an alternative to mean-variance optimization that ignores expected returns entirely – instead it allocates weights so that **each asset contributes equally to total portfolio risk**.

### Why Risk Parity?
Mean-variance optimization is highly sensitive to small changes in expected return estimates (which are notoriously hard to forecast). Risk Parity sidesteps this by relying **only on the covariance matrix**, which is estimated more reliably.

**Bridgewater's All Weather Fund** and many pension funds use Risk Parity as their core allocation framework.

### Risk Contribution:
The **marginal risk contribution** of asset $i$ is:
$$MRC_i = \frac{(\Sigma w)_i}{\sigma_p}$$

The **total risk contribution** of asset $i$ is:
$$TRC_i = w_i \cdot MRC_i$$

Risk Parity solves for $w$ such that $TRC_i = \sigma_p / N$ for all $i$.

In [ ]:
def risk_contributions(w, cov):
    """Compute the risk contribution of each asset."""
    vol    = portfolio_vol(w, cov)
    mrc    = (cov @ w) / vol   # marginal risk contribution vector
    trc    = w * mrc           # total risk contribution = weight × marginal
    return trc

def risk_parity_objective(w, cov):
    """
    Objective: minimise sum of squared differences between
    all pairs of risk contributions → equal risk contribution.
    """
    trc = risk_contributions(w, cov)
    # Penalise any deviation from equal risk sharing
    target = trc.sum() / len(w)
    return np.sum((trc - target) ** 2)

def find_risk_parity(cov):
    n  = cov.shape[0]
    w0 = np.ones(n) / n
    constraints = [{'type': 'eq', 'fun': lambda w: w.sum() - 1}]
    bounds = [(0.001, 1.0)] * n   # small lower bound for numerical stability
    result = minimize(
        fun=risk_parity_objective,
        args=(cov,),
        x0=w0, method='SLSQP',
        bounds=bounds, constraints=constraints,
        options={'ftol': 1e-14, 'maxiter': 2000}
    )
    return result.x

w_rp = find_risk_parity(cov_hat)

print('Risk Parity Portfolio – Weight & Risk Contribution Breakdown:')
trc_rp = risk_contributions(w_rp, cov_hat)
df_rp  = pd.DataFrame({
    'Asset':              assets,
    'Weight':             w_rp,
    'Risk Contribution':  trc_rp,
    '% of Total Risk':    trc_rp / trc_rp.sum() * 100
})
print(df_rp.to_string(index=False, float_format=lambda x: f'{x:.4f}'))
print()
portfolio_summary(w_rp, mu_hat, cov_hat, 'Risk Parity Portfolio')

## Step 7: The Efficient Frontier Plot

This is the central visualisation of Modern Portfolio Theory. It shows:
- The **Efficient Frontier** curve (best possible risk/return combinations)
- The **Capital Market Line (CML)** – the line from the risk-free rate through the Tangency Portfolio
- Individual assets plotted in risk-return space
- All four key portfolios highlighted

The colour gradient on the frontier represents the **Sharpe Ratio** – darker = better risk-adjusted return.

In [ ]:
# ── Sharpe Ratios along the Frontier ─────────────────────────────────────────
frontier_sharpes = (frontier_returns - RISK_FREE_RATE) / frontier_vols

fig, ax = plt.subplots(figsize=(12, 7))

# ── Efficient Frontier (coloured by Sharpe Ratio) ────────────────────────────
sc = ax.scatter(
    frontier_vols * 100, frontier_returns * 100,
    c=frontier_sharpes, cmap='viridis', s=15, zorder=3, alpha=0.85
)
cbar = plt.colorbar(sc, ax=ax)
cbar.set_label('Sharpe Ratio', fontsize=11)

# ── Individual Assets ─────────────────────────────────────────────────────────
asset_vols    = np.sqrt(np.diag(cov_hat)) * 100
asset_returns = mu_hat * 100
ax.scatter(asset_vols, asset_returns, color='#555555', s=60, zorder=5, marker='D')
for i, name in enumerate(assets):
    ax.annotate(name, (asset_vols[i], asset_returns[i]),
                textcoords='offset points', xytext=(6, 3), fontsize=8.5, color='#333')

# ── Capital Market Line ───────────────────────────────────────────────────────
msr_vol = portfolio_vol(w_msr, cov_hat)
msr_ret = portfolio_return(w_msr, mu_hat)
cml_x   = np.linspace(0, msr_vol * 1.4, 100)
slope   = (msr_ret - RISK_FREE_RATE) / msr_vol
cml_y   = RISK_FREE_RATE + slope * cml_x
ax.plot(cml_x * 100, cml_y * 100, 'k--', lw=1.5, alpha=0.6, label='Capital Market Line')

# ── Risk-Free Rate ────────────────────────────────────────────────────────────
ax.scatter([0], [RISK_FREE_RATE * 100], color='black', s=80, zorder=6, marker='*')
ax.annotate(f'Risk-Free\n({RISK_FREE_RATE*100:.1f}%)', (0, RISK_FREE_RATE * 100),
            textcoords='offset points', xytext=(6, -14), fontsize=8.5)

# ── Key Portfolios ────────────────────────────────────────────────────────────
key_portfolios = [
    (w_mvp,   '#2ecc71', 'MVP',            '^', 120),
    (w_msr,   '#e74c3c', 'Max Sharpe',     '*', 200),
    (w_equal, '#3498db', 'Equal Weight',   's',  90),
    (w_rp,    '#9b59b6', 'Risk Parity',    'P', 120),
]
for w, color, label, marker, size in key_portfolios:
    v = portfolio_vol(w, cov_hat) * 100
    r = portfolio_return(w, mu_hat) * 100
    ax.scatter(v, r, color=color, s=size, zorder=7, marker=marker, label=label,
               edgecolors='white', linewidths=0.8)

ax.set_xlabel('Annualised Volatility (%)', fontsize=12)
ax.set_ylabel('Annualised Expected Return (%)', fontsize=12)
ax.set_title('Efficient Frontier – 8-Asset European Equity Portfolio',
             fontsize=14, fontweight='bold')
ax.legend(fontsize=10, loc='lower right')
plt.tight_layout()
plt.savefig('efficient_frontier.png', dpi=150, bbox_inches='tight')
plt.show()

## Step 8: Portfolio Weight Allocation Comparison

Visualising the weight allocation across all four strategies reveals how differently each approach distributes capital across assets. This is a key output for portfolio managers when communicating allocation decisions to clients.

In [ ]:
strategies = {
    'Min Variance':  w_mvp,
    'Max Sharpe':    w_msr,
    'Equal Weight':  w_equal,
    'Risk Parity':   w_rp,
}

fig, axes = plt.subplots(2, 2, figsize=(14, 9))
colors_bar = ['#4a90d9', '#e67e22', '#2ecc71', '#9b59b6',
               '#e74c3c', '#1abc9c', '#f39c12', '#8e44ad']

for ax, (name, w) in zip(axes.flat, strategies.items()):
    bars = ax.bar(assets, w * 100, color=colors_bar, edgecolor='white', linewidth=0.5)
    ax.bar_label(bars, fmt='%.1f%%', padding=2, fontsize=8)
    sr  = sharpe_ratio(w, mu_hat, cov_hat)
    ret = portfolio_return(w, mu_hat)
    vol = portfolio_vol(w, cov_hat)
    ax.set_title(f'{name}\nReturn: {ret*100:.2f}%  |  Vol: {vol*100:.2f}%  |  Sharpe: {sr:.3f}',
                 fontweight='bold', fontsize=10)
    ax.set_ylabel('Weight (%)')
    ax.set_ylim(0, max(w * 100) * 1.3)
    ax.tick_params(axis='x', rotation=30)

plt.suptitle('Portfolio Weight Allocation by Strategy', fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('weight_allocation.png', dpi=150, bbox_inches='tight')
plt.show()

## Step 9: Simulated Portfolio Performance (Backtest)

We simulate how each strategy would have performed over the historical period by applying the (static) optimized weights to the daily return history. This is a **static backtest** – in practice, portfolios are rebalanced periodically (monthly, quarterly).

We compute and compare:
- **Cumulative return** (wealth growth curve)
- **Maximum Drawdown** (worst peak-to-trough loss)
- **Annualised Sharpe Ratio** realized over the period
- **Calmar Ratio** = Annualised Return / |Max Drawdown| – used by hedge funds

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(13, 10))

palette = {
    'Min Variance': '#2ecc71',
    'Max Sharpe':   '#e74c3c',
    'Equal Weight': '#3498db',
    'Risk Parity':  '#9b59b6',
}

perf_summary = []

for name, w in strategies.items():
    port_daily   = df_returns.values @ w               # daily portfolio returns
    cum_returns  = (1 + port_daily).cumprod()          # cumulative wealth factor
    mdd          = max_drawdown(cum_returns)
    ann_return   = cum_returns[-1] ** (TRADING_DAYS / N_DAYS_HISTORY) - 1
    ann_vol_real = port_daily.std() * np.sqrt(TRADING_DAYS)
    sharpe_real  = (ann_return - RISK_FREE_RATE) / ann_vol_real
    calmar       = ann_return / abs(mdd) if mdd != 0 else np.inf

    perf_summary.append({
        'Strategy':       name,
        'Ann. Return':    ann_return,
        'Ann. Vol':       ann_vol_real,
        'Sharpe':         sharpe_real,
        'Max Drawdown':   mdd,
        'Calmar Ratio':   calmar,
        'Final Value (€)': PORTFOLIO_VALUE * cum_returns[-1]
    })

    color = palette[name]
    axes[0].plot(cum_returns * PORTFOLIO_VALUE / 1e6,
                 label=name, color=color, lw=2)

    # Drawdown
    rolling_max  = np.maximum.accumulate(cum_returns)
    drawdown_series = (cum_returns - rolling_max) / rolling_max * 100
    axes[1].fill_between(range(len(drawdown_series)), drawdown_series, 0,
                         alpha=0.25, color=color)
    axes[1].plot(drawdown_series, color=color, lw=1.2, label=name)

axes[0].set_title('Cumulative Portfolio Value (€M)', fontweight='bold', fontsize=12)
axes[0].set_ylabel('Portfolio Value (€M)')
axes[0].yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'€{x:.2f}M'))
axes[0].legend(fontsize=10)

axes[1].set_title('Portfolio Drawdown (%)', fontweight='bold', fontsize=12)
axes[1].set_ylabel('Drawdown (%)')
axes[1].set_xlabel('Trading Days')
axes[1].legend(fontsize=10)

plt.tight_layout()
plt.savefig('performance.png', dpi=150, bbox_inches='tight')
plt.show()

# ── Performance Table ────────────────────────────────────────────────────────
df_perf = pd.DataFrame(perf_summary)
print('\nPerformance Summary:')
print(df_perf[['Strategy','Ann. Return','Ann. Vol','Sharpe','Max Drawdown','Calmar Ratio','Final Value (€)']]
      .to_string(index=False, float_format=lambda x: f'{x:.4f}'))

## Step 10: Risk Contribution Comparison

This plot compares how each strategy distributes **risk** (not just capital) across assets. It directly illustrates the key difference between Risk Parity and the other approaches:
- **Mean-Variance / Max Sharpe** portfolios concentrate risk in a few assets
- **Risk Parity** achieves near-equal risk contributions from each asset

For a risk manager, this distinction is crucial – a portfolio where one asset drives 60% of risk is vulnerable to single-name events.

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(16, 5))

for ax, (name, w) in zip(axes, strategies.items()):
    trc     = risk_contributions(w, cov_hat)
    trc_pct = trc / trc.sum() * 100

    bars = ax.bar(assets, trc_pct,
                  color=colors_bar, edgecolor='white', linewidth=0.5)
    ax.axhline(100 / n, color='red', ls='--', lw=1.2,
               label=f'Equal ({100/n:.1f}%)')
    ax.set_title(name, fontweight='bold', fontsize=10)
    ax.set_ylabel('% of Total Risk')
    ax.set_ylim(0, trc_pct.max() * 1.3)
    ax.tick_params(axis='x', rotation=40)
    ax.legend(fontsize=8)

plt.suptitle('Risk Contribution by Asset and Strategy',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('risk_contributions.png', dpi=150, bbox_inches='tight')
plt.show()

## Summary & Key Findings

This notebook implemented a complete **Modern Portfolio Theory framework** covering four core allocation strategies:

| Strategy | Return | Volatility | Sharpe | Max DD | Best suited for |
|----------|--------|------------|--------|--------|-----------------|
| Min Variance | Lowest | Lowest | Moderate | Smallest | Risk-averse mandates |
| Max Sharpe | Moderate | Moderate | Highest | Moderate | General institutional use |
| Equal Weight | Moderate | Moderate | Moderate | Moderate | Simple benchmark |
| Risk Parity | Moderate | Low-Moderate | Good | Small | Balanced mandates |

### Key Insights for Portfolio Risk Management:

1. **The Efficient Frontier is a constraint, not a suggestion.** Any portfolio below the frontier is *dominated* – it offers less return for the same risk. Identifying dominated portfolios and replacing them with frontier equivalents is pure alpha.

2. **Max Sharpe ≠ Max Return.** The Tangency Portfolio maximises risk-adjusted return – it sacrifices some raw return for meaningfully lower volatility. For leverage-constrained investors, this is optimal.

3. **Risk Parity avoids return-estimation error.** Expected returns are notoriously difficult to forecast. Risk Parity's reliance on the covariance matrix alone makes it more robust to estimation error – which is why it is widely used in practice.

4. **Drawdown matters as much as volatility.** The Calmar Ratio (return / max drawdown) is often more meaningful to clients than the Sharpe Ratio – a portfolio with low drawdown is psychologically easier to hold through downturns.

### Extensions for Production Use:
- **Transaction cost modeling**: Rebalancing incurs costs; optimal rebalancing frequency must be determined
- **Robust optimization**: Replace point estimates with uncertainty sets (Ellipsoidal uncertainty, Black-Litterman)
- **Factor models**: Decompose returns into systematic factors (market, size, value) to reduce estimation error
- **Dynamic allocation**: Incorporate GARCH volatility forecasts for time-varying covariance
- **ESG constraints**: Add constraint for minimum ESG score (increasingly standard in EU mandates)
- **Multi-asset extension**: Bonds, commodities, FX, alternatives alongside equities